In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
from statsforecast import StatsForecast
from statsforecast.models import AutoETS
from tqdm import tqdm
import os

os.environ["NIXTLA_ID_AS_COL"] = "False"

In [3]:
# Load data files
data = pd.read_csv("C:/Data/M5store_department.csv")

In [9]:
# Define the MASE function
def mase(y, y_pred, y_train, seasonality=1):
    """
    Calculate Mean Absolute Scaled Error (MASE)
    y: Actual values
    y_pred: Predicted values
    y_train: Training data for scaling factor
    seasonality: Seasonal period for naive forecasting
    """
    mae = np.mean(np.abs(y - y_pred))
    naive_forecast_errors = np.abs(y_train[seasonality:] - y_train[:-seasonality])
    scaling_factor = np.mean(naive_forecast_errors)
    return mae / scaling_factor

# Leave-One-Out Cross-Validation for last m points
def leave_one_out_cv_last_m_with_naive(df, m, h, model, seasonality=7):
    """
    Perform Leave-One-Out Cross-Validation on the last m data points with both AutoETS and naive forecasts.
    
    df: DataFrame with columns 'ds', 'y', and 'unique_id'.
    m: Number of data points from the end of the dataset for cross-validation.
    h: Number of steps ahead for forecasting.
    model: Model object with .fit() and .predict() methods.
    seasonality: Seasonal period for naive forecasting.
    
    Returns:
    - MASE values for AutoETS and naive forecasts.
    """
    errors_autoets = []  # Store actual and predicted values for AutoETS
    errors_naive = []    # Store actual and predicted values for naive forecast
    start_index = len(df) - m  # Start index for cross-validation

    for i in range(start_index, len(df) - h + 1):
        # Training and test split
        train_subset = df.iloc[:i]  # Use all points up to the current fold
        test_subset = df.iloc[i:i + h]

        # Fit the model on the training subset
        autoets = model.fit(train_subset['y'].values)

        # Predict for the test subset using AutoETS
        y_hat_autoets = autoets.predict(h=h).get("mean")
        errors_autoets.extend(zip(test_subset['y'].values, y_hat_autoets))

        # Calculate naive forecast
        y_hat_naive = train_subset['y'].iloc[-seasonality:].values.tolist() * h
        y_hat_naive = y_hat_naive[:h]  # Ensure forecast length matches h
        errors_naive.extend(zip(test_subset['y'].values, y_hat_naive))

    # Calculate MASE for AutoETS
    actual_autoets = np.array([e[0] for e in errors_autoets])
    predicted_autoets = np.array([e[1] for e in errors_autoets])
    train_series = df['y'].values  # Full training series for scaling
    mase_autoets = mase(actual_autoets, predicted_autoets, train_series)

    # Calculate MASE for naive forecast
    actual_naive = np.array([e[0] for e in errors_naive])
    predicted_naive = np.array([e[1] for e in errors_naive])
    mase_naive = mase(actual_naive, predicted_naive, train_series)

    return mase_autoets, mase_naive

In [10]:
# Parameters
unique_store_ids = data['store_dept_id'].unique()
m = 28  # Only consider the last m points for cross-validation
h = 1  # Number of steps ahead for forecasting
seasonality = 7  # Weekly seasonality for daily data

results = []

# Loop through each store_id with progress tracking
for store_dept_id in tqdm(unique_store_ids, desc="Processing all store_dept_id series"):
    df = data.loc[data['store_dept_id'] == store_dept_id, ['d', 'revenue', 'store_dept_id']]
    df = df.rename(columns={'d': 'ds', 'revenue': 'y', 'store_dept_id': 'unique_id'})
    
    model = AutoETS(model=["Z", "Z", "Z"], alias="AutoETS", damped=True, season_length=seasonality)
    mase_autoets, mase_naive = leave_one_out_cv_last_m_with_naive(df, m, h, model, seasonality)
    results.append({"store_dept_id": store_dept_id, "AutoETS_MASE": mase_autoets, "Naive_MASE": mase_naive})

# Create a summary table
summary_table = pd.DataFrame(results)

# Calculate average MASE for both AutoETS and naive forecasts
average_autoets_mase = summary_table["AutoETS_MASE"].mean()
average_naive_mase = summary_table["Naive_MASE"].mean()

print("Summary Table of MASE for Each Series:")
print(summary_table)
print(f"\nAverage AutoETS MASE across all series: {average_autoets_mase}")
print(f"Average Naive MASE across all series: {average_naive_mase}")

Processing all store_dept_id series: 100%|█████████████████████████████████████████████| 70/70 [10:37<00:00,  9.10s/it]

Summary Table of MASE for Each Series:
       store_dept_id  AutoETS_MASE  Naive_MASE
0       CA_1_FOODS_1      0.853588    0.975382
1       CA_1_FOODS_2      0.706487    0.991145
2       CA_1_FOODS_3      0.429324    0.502431
3     CA_1_HOBBIES_1      0.827673    1.293331
4     CA_1_HOBBIES_2      0.707072    0.980284
..               ...           ...         ...
65      WI_3_FOODS_3      0.725154    1.172751
66    WI_3_HOBBIES_1      0.670144    0.825089
67    WI_3_HOBBIES_2      0.940957    0.998925
68  WI_3_HOUSEHOLD_1      0.776357    1.056118
69  WI_3_HOUSEHOLD_2      0.780185    0.934251

[70 rows x 3 columns]

Average AutoETS MASE across all series: 0.8368331832384662
Average Naive MASE across all series: 1.1644184722695485


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import warnings; warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 110})

data['ds_parsed'] = pd.to_datetime(data['d'])
data['store'] = data['store_dept_id'].apply(lambda x: '_'.join(x.split('_')[:2]))
data['dept_cat'] = data['store_dept_id'].apply(lambda x: '_'.join(x.split('_')[2:]))
store_ids = sorted(data['store'].unique())
dept_cats = sorted(data['dept_cat'].unique())
pal = plt.cm.tab10(np.linspace(0, 1, len(store_ids)))

# ── Easy: MASE Distribution by Department Category ────────────────────────────
summary_table['store'] = summary_table['store_dept_id'].apply(lambda x: '_'.join(x.split('_')[:2]))
summary_table['dept_cat'] = summary_table['store_dept_id'].apply(lambda x: '_'.join(x.split('_')[2:]))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
dept_mase = summary_table.groupby('dept_cat')['AutoETS_MASE'].mean().sort_values()
c_bars = ['#e74c3c' if v > 1 else '#2ecc71' for v in dept_mase.values]
axes[0].barh(range(len(dept_mase)), dept_mase.values, color=c_bars, edgecolor='white')
axes[0].axvline(1.0, color='black', lw=1.5, linestyle='--', label='Naive = 1')
axes[0].set_yticks(range(len(dept_mase))); axes[0].set_yticklabels(dept_mase.index, fontsize=9)
axes[0].set_title('Avg MASE by Department Category', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Average MASE'); axes[0].legend(); axes[0].spines[['top','right']].set_visible(False)

store_mase = summary_table.groupby('store')['AutoETS_MASE'].mean().sort_values()
c_bars2 = ['#e74c3c' if v > 1 else '#2ecc71' for v in store_mase.values]
axes[1].barh(range(len(store_mase)), store_mase.values, color=c_bars2, edgecolor='white')
axes[1].axvline(1.0, color='black', lw=1.5, linestyle='--')
axes[1].set_yticks(range(len(store_mase))); axes[1].set_yticklabels(store_mase.index, fontsize=9)
axes[1].set_title('Avg MASE by Store (Dept-Level)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Average MASE'); axes[1].spines[['top','right']].set_visible(False)
plt.suptitle('AutoETS MASE at Department Level', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

# ── Medium: Revenue Time Series — 3 Depts per Store ──────────────────────────
sample_depts = [d for d in data['store_dept_id'].unique()[:12]]
fig, axes = plt.subplots(4, 3, figsize=(18, 14))
colors_d = plt.cm.Set2(np.linspace(0, 1, 12))
for ax, dept_id, color in zip(axes.flat, sample_depts, colors_d):
    sd = data[data['store_dept_id']==dept_id].sort_values('ds_parsed').tail(90)
    ax.plot(sd['ds_parsed'], sd['revenue'], lw=1.5, color=color)
    ax.fill_between(sd['ds_parsed'], sd['revenue'], alpha=0.15, color=color)
    ax.set_title(dept_id, fontsize=8, fontweight='bold')
    ax.tick_params(axis='x', rotation=30, labelsize=6); ax.tick_params(axis='y', labelsize=6)
    ax.spines[['top','right']].set_visible(False)
plt.suptitle('Department Revenue Time Series — Last 90 Days (12 Depts)', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

# ── Medium: MASE Heatmap — Store × Department Category ───────────────────────
mase_heat = summary_table.pivot(index='store', columns='dept_cat', values='AutoETS_MASE')
fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(mase_heat, cmap='RdYlGn_r', center=1.0, linewidths=0.4, linecolor='white',
            annot=True, fmt='.3f', annot_kws={'size': 9},
            cbar_kws={'label': 'MASE (red > 1 = worse than naive)'}, ax=ax)
ax.set_title('MASE Heatmap — Store × Department Category\n(Green < 1 = beats naive baseline)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Department Category'); ax.set_ylabel('Store')
plt.tight_layout(); plt.show()

# ── Hard: Department Revenue Share Stacked Area per Store ─────────────────────
fig, axes = plt.subplots(5, 2, figsize=(16, 18))
for ax, store, color in zip(axes.flat, store_ids, pal):
    sd = data[data['store']==store].copy()
    pivot = sd.pivot_table(index='ds_parsed', columns='dept_cat', values='revenue', aggfunc='sum').fillna(0)
    pivot_monthly = pivot.resample('W').sum()
    ax.stackplot(pivot_monthly.index, pivot_monthly.T, labels=pivot_monthly.columns,
                 alpha=0.85, colors=plt.cm.Set3(np.linspace(0, 1, len(pivot_monthly.columns))))
    ax.set_title(f'{store}', fontsize=10, fontweight='bold')
    ax.tick_params(axis='x', rotation=30, labelsize=6); ax.tick_params(axis='y', labelsize=7)
    ax.spines[['top','right']].set_visible(False)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v,_: f'${v/1000:.0f}k'))
handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=3, fontsize=9,
           bbox_to_anchor=(0.5, -0.02))
plt.suptitle('Weekly Revenue Stacked by Department Category — All Stores', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

# ── Hard: MASE Violin by Department Category + Store Scatter ──────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 7))
sns.violinplot(data=summary_table, x='dept_cat', y='AutoETS_MASE',
               palette='Set2', inner='box', cut=0, ax=axes[0])
axes[0].axhline(1.0, color='red', lw=1.8, linestyle='--', label='Naive baseline')
axes[0].set_title('MASE Distribution by Department Category', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Department Category'); axes[0].set_ylabel('MASE')
axes[0].legend(fontsize=10); axes[0].tick_params(axis='x', rotation=20)

naive_v = summary_table['Naive_MASE']
ets_v = summary_table['AutoETS_MASE']
axes[1].scatter(naive_v, ets_v, c=plt.cm.tab10(pd.factorize(summary_table['store'])[0] / 10),
                s=60, alpha=0.8, edgecolors='white', lw=0.5)
axes[1].plot([0, 2.5], [0, 2.5], 'k--', lw=1.5, label='ETS = Naive')
axes[1].set_xlabel('Naive MASE'); axes[1].set_ylabel('AutoETS MASE')
axes[1].set_title('AutoETS vs Naive MASE — All 70 Series\n(Below diagonal = ETS beats naive)',
                  fontsize=12, fontweight='bold')
for store, color in zip(store_ids, pal):
    axes[1].scatter([], [], color=color, label=store, s=60)
axes[1].legend(fontsize=8, ncol=2)
plt.tight_layout(); plt.show()